In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
# Load processed dataset

DATA_PATH = "../DATA/processed/personality_model_data.csv" 
df = pd.read_csv(DATA_PATH)
# Separate features and targets

X = df.iloc[:, :50]
y = df[["E", "A", "C", "O", "N"]]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Split features and targets into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Verify the split
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
# Check missing values in training features
missing = X_train.isna().sum()

print("Total missing values:", missing.sum())
print("\nColumns with missing values:")
print(missing[missing > 0].sort_values(ascending=False))

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

# Build baseline pipeline
model = make_pipeline(
    SimpleImputer(strategy="median"),
    LinearRegression()
)

# Train the model
model.fit(X_train, y_train)

print("Baseline model trained successfully!")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Generate predictions on unseen test data
y_pred = model.predict(X_test)

# Evaluate all five OCEAN targets together
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = model.score(X_test, y_test)

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

In [ ]:
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

results = []

for i, trait in enumerate(y.columns):
    results.append({
        "Trait": trait,
        "MAE": mean_absolute_error(y_test.iloc[:, i], y_pred[:, i]),
        "RMSE": np.sqrt(mean_squared_error(y_test.iloc[:, i], y_pred[:, i])),
        "R2": r2_score(y_test.iloc[:, i], y_pred[:, i])
    })

results_df = pd.DataFrame(results).round(4)
print(results_df)

In [ ]:
# Save baseline evaluation results
results_df.to_csv(
    "../DATA/processed/baseline_results.csv",
    index=False
)

print("Baseline results saved successfully!")

In [ ]:
from sklearn.model_selection import cross_val_score

# Use a reproducible 50,000-row training sample
X_cv = X_train.sample(n=50_000, random_state=42)
y_cv = y_train.loc[X_cv.index]

# Evaluate the pipeline using 3-fold cross-validation
cv_scores = cross_val_score(
    model,
    X_cv,
    y_cv,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

# Convert negative MAE scores into positive errors
cv_mae = -cv_scores

print("Fold MAE scores:", cv_mae.round(5))
print("Mean CV MAE:", round(cv_mae.mean(), 5))
print("Std CV MAE:", round(cv_mae.std(), 5))

NOW TRYING DIFFERENT ENSEMBLE  TECHNIQUES


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


# Use a manageable training sample
X_rf = X_train.sample(n=20000, random_state=42)
y_rf = y_train.loc[X_rf.index]

# Train Random Forest with missing-value imputation
rf_model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
)

rf_model.fit(X_rf, y_rf)

# Evaluate on the existing test set
rf_pred = rf_model.predict(X_test)

print("Random Forest MAE:", round(mean_absolute_error(y_test, rf_pred), 5))

 WE ARE TRYING THE RANDOM FOREST USING SMALL AMOUNT OF DATA TO SEE THE COMPUTATIONAL COST FIRST THEN DECIDE WHAT TO DO LATER 

In [ ]:

from sklearn.metrics import mean_absolute_error

# Compare both models on the same test data
comparison = pd.DataFrame({
    "Trait": y.columns,
    "Linear Regression MAE": [
        mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        for i in range(len(y.columns))
    ],
    "Random Forest MAE": [
        mean_absolute_error(y_test.iloc[:, i], rf_pred[:, i])
        for i in range(len(y.columns))
    ]
})

print(comparison.round(5).to_string(index=False))

so here we are seeing that the random forest's performance is not that good  compared to thelinear regression 
so well check  another ensemble leaning technique 

In [ ]:
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import make_pipeline
X_ada = X_train.sample(n=20_000, random_state=42)
y_ada = y_train.loc[X_ada.index]
ada_model = make_pipeline(
    SimpleImputer(strategy="median"),
    MultiOutputRegressor(
        AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=4),
            n_estimators=50,
            learning_rate=0.1,
            random_state=42
        )
    )
)

ada_model.fit(X_ada, y_ada)

ada_pred = ada_model.predict(X_test)

print("AdaBoost MAE:", round(mean_absolute_error(y_test, ada_pred), 5))

its better then random forest but still not better  than linear regression, so now using gradient boosting regressor 

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

gb_model = make_pipeline(
    SimpleImputer(strategy="median"),
    MultiOutputRegressor(
        GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42
        )
    )
)

gb_model.fit(X_ada, y_ada)
gb_pred = gb_model.predict(X_test)

print("Gradient Boosting MAE:", round(mean_absolute_error(y_test, gb_pred), 5))

WE'VE TRIED 3 ENSEMBKE TECHNIQUES BUT STILL LINEAR REGRESSION GAVE THE BEST PREDICTIONS 

In [ ]:
# Verify the fitted baseline pipeline
print("Pipeline:", model)
print("Input features:", X_train.shape[1])
print("Output targets:", y_train.shape[1])
print("Fitted:", hasattr(model[-1], "coef_"))

In [ ]:
import joblib
from pathlib import Path

# Create the model directory if it doesn't exist
model_dir = Path("../MODELS")
model_dir.mkdir(parents=True, exist_ok=True)

# Save the complete fitted pipeline
model_path = model_dir / "personality_baseline.joblib"
joblib.dump(model, model_path)

print("Model saved successfully!")
print("Location:", model_path.resolve())